# Phase 4 - Coordinate-system normalisation

This notebook normalises every Phase 3 GIS source to GDA2020 / MGA zone 56 (`EPSG:7856`). The ELVIS DEM and LAZ use the explicitly selected EPSG:8447 conformal-and-distortion grid operation. Their AHD Z values are not subjected to a vertical transformation.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'pyproject.toml').exists():
    project_root = project_root.parent
if not (project_root / 'pyproject.toml').exists():
    raise RuntimeError('Could not locate the project root containing pyproject.toml')

src_path = str(project_root / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import pandas as pd
from IPython.display import JSON, Image, display

from uqgems.phase4 import run_phase4

project_root

## Run and validate normalisation

The initial run creates the transformed products. Later runs reuse them only when the source hashes, datum grid and transformation configuration still match the manifest.

In [ ]:
summary = run_phase4(project_root)
print(f"Phase 4 status: {summary['status']}")
print(f"Cached transformed products reused: {summary['cache_reused']}")
display(JSON(summary['checks'], expanded=True))

## Datum transformation

The figure shows the real coordinate change at a representative point. This is not an arbitrary display offset.

In [ ]:
display(Image(filename=str(project_root / summary['outputs']['datum_shift_figure'])))
display(JSON(summary['datum_shift_at_pilot'], expanded=True))
display(JSON(summary['transformation'], expanded=False))

## Normalisation register and numerical checks

The register distinguishes horizontal and vertical operations and records source/output hashes. DEM values are compared after interpolation; sampled LAZ coordinates, Z values and classifications are compared point-for-point.

In [ ]:
register = pd.read_csv(project_root / summary['outputs']['normalisation_register'])
display(register)
display(pd.DataFrame({
    'DEM comparison': summary['dem_elevation_comparison'],
    'LAZ sample comparison': summary['laz_sample_comparison'],
}).T)

## Phase 4 quality gate

Passing means every modelling input has a known CRS, every normalised GIS layer uses `EPSG:7856`, AHD handling is explicit, and the local rendering translation round-trips correctly. Outlier removal, point-density analysis and cropping remain Phase 5 tasks.